In [2]:
!pip install apscheduler



   ---------------------------------------- 2/2 [apscheduler]



In [ ]:
import asyncio
from apscheduler.schedulers.asyncio import AsyncIOScheduler
from datetime import datetime

# Dummy async functions simulating your jobs
async def reindex():
    print(f"[{datetime.now()}] Starting re-indexing...")
    await asyncio.sleep(5)  # Simulate time-consuming reindexing
    print(f"[{datetime.now()}] Re-indexing completed.")

async def rlhf_retraining():
    print(f"[{datetime.now()}] Starting RLHF retraining...")
    await asyncio.sleep(10)  # Simulate long retraining
    print(f"[{datetime.now()}] RLHF retraining completed.")

# Wrapper to run async functions from APScheduler
def run_async_job(coro):
    asyncio.create_task(coro)

async def main():
    scheduler = AsyncIOScheduler()

    # Schedule reindexing every 6 hours
    scheduler.add_job(run_async_job, 'interval', hours=6, args=[reindex()])

    # Schedule RLHF retraining every day at 2:00 AM
    scheduler.add_job(run_async_job, 'cron', hour=2, minute=0, args=[rlhf_retraining()])

    scheduler.start()

    print(f"[{datetime.now()}] Scheduler started. Running jobs asynchronously.")

    # Keep the program running to allow scheduled jobs to run
    while True:
        await asyncio.sleep(3600)  # sleep 1 hour, can be any long time

if __name__ == "__main__":
    await main()



[2025-07-28 18:19:15.815182] Scheduler started. Running jobs asynchronously.


In [ ]:
import asyncio
import logging
from apscheduler.schedulers.asyncio import AsyncIOScheduler
from datetime import datetime

# Setup logger
logging.basicConfig(
    level=logging.INFO,
    format='[%(asctime)s] %(levelname)s: %(message)s',
    handlers=[
        logging.FileHandler("background_jobs.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# Placeholder: Your actual reward model retraining function
async def reward_model_retraining():
    logger.info("Reward model retraining started.")
    try:
        # Replace below with your actual RLHF training code, e.g.,
        # await my_reward_model.train()
        await asyncio.sleep(10)  # Simulate training time
        logger.info("Reward model retraining completed successfully.")
    except Exception as e:
        logger.error(f"Error during reward model retraining: {e}")
        raise

# Placeholder: Your reindexing function
async def reindex():
    logger.info("Re-indexing started.")
    try:
        # Replace below with your actual re-indexing code
        await asyncio.sleep(5)  # Simulate re-indexing
        logger.info("Re-indexing completed successfully.")
    except Exception as e:
        logger.error(f"Error during re-indexing: {e}")
        raise

# Helper to run async job and catch errors
def run_async_job(coro):
    async def wrapper():
        try:
            await coro
        except Exception as e:
            logger.error(f"Unhandled exception in async job: {e}")
    asyncio.create_task(wrapper())

async def main():
    scheduler = AsyncIOScheduler()

    # Schedule re-indexing every 6 hours
    scheduler.add_job(run_async_job, 'interval', hours=6, args=[reindex()])

    # Schedule RLHF retraining every day at 2:00 AM
    scheduler.add_job(run_async_job, 'cron', hour=2, minute=0, args=[reward_model_retraining()])

    scheduler.start()

    logger.info("Scheduler started. Waiting for scheduled jobs to run.")

    # Keep program alive
    while True:
        await asyncio.sleep(3600)

if __name__ == "__main__":
    await main()


[2025-07-28 18:22:19,255] INFO: Adding job tentatively -- it will be properly scheduled when the scheduler starts
[2025-07-28 18:22:19,259] INFO: Adding job tentatively -- it will be properly scheduled when the scheduler starts
[2025-07-28 18:22:19,261] INFO: Added job "run_async_job" to job store "default"
[2025-07-28 18:22:19,262] INFO: Added job "run_async_job" to job store "default"
[2025-07-28 18:22:19,263] INFO: Scheduler started
[2025-07-28 18:22:19,264] INFO: Scheduler started. Waiting for scheduled jobs to run.


In [1]:
import asyncio

def sync_train_reward_model():
    # your existing reward model training code here
    pass

async def reward_model_retraining():
    logger.info("Reward model retraining started.")
    try:
        await asyncio.to_thread(sync_train_reward_model)
        logger.info("Reward model retraining completed successfully.")
    except Exception as e:
        logger.error(f"Error during reward model retraining: {e}")
        raise


In [2]:
import smtplib
from email.message import EmailMessage

# Configure your email details here
SMTP_SERVER = 'smtp.gmail.com'
SMTP_PORT = 587
SMTP_USER = 'your_email@gmail.com'
SMTP_PASSWORD = 'your_app_password'  # Use app password, NOT your actual Gmail password
ALERT_RECIPIENT = 'recipient_email@example.com'

def send_error_email(subject: str, body: str):
    msg = EmailMessage()
    msg['From'] = SMTP_USER
    msg['To'] = ALERT_RECIPIENT
    msg['Subject'] = subject
    msg.set_content(body)

    try:
        with smtplib.SMTP(SMTP_SERVER, SMTP_PORT) as smtp:
            smtp.starttls()
            smtp.login(SMTP_USER, SMTP_PASSWORD)
            smtp.send_message(msg)
        logger.info("Sent error alert email.")
    except Exception as e:
        logger.error(f"Failed to send error alert email: {e}")


In [3]:
def run_async_job(coro):
    async def wrapper():
        try:
            await coro
        except Exception as e:
            error_msg = f"Unhandled exception in async job: {e}"
            logger.error(error_msg)
            send_error_email("Background Job Failure Alert", error_msg)
    asyncio.create_task(wrapper())


In [4]:
!pip install prometheus_client


In [5]:
from prometheus_client import start_http_server, Counter

job_success_counter = Counter('background_jobs_success_total', 'Number of successful background jobs', ['job_name'])
job_failure_counter = Counter('background_jobs_failure_total', 'Number of failed background jobs', ['job_name'])


In [6]:
async def reward_model_retraining():
    job_name = 'reward_model_retraining'
    logger.info(f"{job_name} started.")
    try:
        await asyncio.sleep(10)  # replace with actual training
        job_success_counter.labels(job_name=job_name).inc()
        logger.info(f"{job_name} completed successfully.")
    except Exception as e:
        job_failure_counter.labels(job_name=job_name).inc()
        logger.error(f"Error during {job_name}: {e}")
        raise

async def reindex():
    job_name = 'reindex'
    logger.info(f"{job_name} started.")
    try:
        await asyncio.sleep(5)  # replace with actual reindexing
        job_success_counter.labels(job_name=job_name).inc()
        logger.info(f"{job_name} completed successfully.")
    except Exception as e:
        job_failure_counter.labels(job_name=job_name).inc()
        logger.error(f"Error during {job_name}: {e}")
        raise


In [7]:
async def main():
    # Start Prometheus metrics server on port 8000
    start_http_server(8000)
    logger.info("Prometheus metrics HTTP server started on port 8000.")

    scheduler = AsyncIOScheduler()
    # ... your scheduled jobs here ...


In [8]:
!pip install fastapi uvicorn


In [9]:
from fastapi import FastAPI, HTTPException
import uvicorn

app = FastAPI()

@app.post("/trigger/{job_name}")
async def trigger_job(job_name: str):
    if job_name == 'reindex':
        run_async_job(reindex())
        return {"status": "Reindexing triggered"}
    elif job_name == 'rlhf_retraining':
        run_async_job(reward_model_retraining())
        return {"status": "RLHF retraining triggered"}
    else:
        raise HTTPException(status_code=404, detail="Job not found")

# Run FastAPI in background alongside scheduler
async def start_api():
    config = uvicorn.Config(app, host="0.0.0.0", port=8080, log_level="info")
    server = uvicorn.Server(config)
    await server.serve()


In [10]:
async def main():
    # Start Prometheus metrics server
    start_http_server(8000)
    logger.info("Prometheus metrics HTTP server started on port 8000.")

    scheduler = AsyncIOScheduler()
    scheduler.add_job(run_async_job, 'interval', hours=6, args=[reindex()])
    scheduler.add_job(run_async_job, 'cron', hour=2, minute=0, args=[reward_model_retraining()])
    scheduler.start()
    logger.info("Scheduler started.")

    # Run API and keep alive concurrently
    await asyncio.gather(
        start_api(),
        keep_alive()
    )

async def keep_alive():
    while True:
        await asyncio.sleep(3600)


In [11]:
import functools
import random

def retry_async(max_retries=3, base_delay=2):
    def decorator(func):
        @functools.wraps(func)
        async def wrapper(*args, **kwargs):
            retries = 0
            while retries < max_retries:
                try:
                    return await func(*args, **kwargs)
                except Exception as e:
                    wait_time = base_delay * (2 ** retries) + random.uniform(0, 1)
                    logger.warning(f"Retry {retries+1}/{max_retries} after error: {e}. Waiting {wait_time:.2f}s.")
                    await asyncio.sleep(wait_time)
                    retries += 1
            logger.error(f"Failed after {max_retries} retries.")
            raise
        return wrapper
    return decorator


In [13]:
@retry_async(max_retries=3)
async def reward_model_retraining():
    # your actual retraining code here
    logger.info("Reward model retraining started.")
    try:
        await asyncio.sleep(10)  # simulate training
        logger.info("Reward model retraining completed successfully.")
    except Exception as e:
        logger.error(f"Error during reward model retraining: {e}")
        raise
